# Problem 2: SaaS Subscription Churn with Google TabFM (KKBox, RAM-Safe Production Notebook)

This notebook solves the problem:

- Predict whether paying subscribers will renew or churn.
- Convert predictions into customer-success actions that can improve adoption and ARR.

This implementation is designed to run **end-to-end without OOM** on typical 16-32 GB RAM machines by:

- Using streaming/lazy feature engineering with Polars.
- Aligning labels and logs to the same month window (`train_v2` + `user_logs_v2`).
- Training TabFM on a stratified context subset while evaluating on larger holdout data.


## 0) Why this design

### Business goal
We treat churn prediction as a retention intervention problem:

- Prioritize high-risk accounts for customer-success outreach.
- Recommend onboarding/adoption workflows for medium-risk users.
- Estimate campaign ROI using expected saved customers and net value.

### Data and temporal alignment
The KKBox `user_logs_v2.csv` file covers **March 2017** only. To avoid missing feature leakage and empty log windows, this notebook uses:

- Labels: `train_v2.csv`
- Transactions: `transactions_v2.csv`
- Logs: `user_logs_v2.csv`
- Members: `members_v3.csv`

### TabFM capability coverage
We train three TabFM variants:

1. `tabfm_default` (robust baseline)
2. `tabfm_ensemble_preset` (official ensemble preset)
3. `tabfm_advanced_custom` (advanced normalization/cross features/calibration options)


## 1) Reproducible setup

Set environment variables before running if needed:

- `TABFM_DEVICE=auto|cpu|cuda`
- `KKBOX_SAMPLE_TRAIN_ROWS=120000`
- `KKBOX_SAMPLE_EVAL_ROWS=120000`
- `TABFM_CONTEXT_MAX_ROWS=1500`
- `TABFM_EVAL_MAX_ROWS=1000`
- `TABFM_FAST_MODE=0|1`
- `KKBOX_MIN_SAMPLE_ROWS=2000`
- `TABFM_CHECKPOINT_PATH=/path/to/pytorch_model.bin` (optional)

License note:
- TabFM code is Apache-2.0.
- Released TabFM weights are non-commercial licensed by Google. Review before commercial use.


In [ ]:
from __future__ import annotations

import json
import os
import random
import shutil
import subprocess
import time
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import torch
from loguru import logger
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TABFM_DEVICE_PREF = os.getenv('TABFM_DEVICE', 'auto').lower().strip()
if TABFM_DEVICE_PREF not in {'auto', 'cpu', 'cuda'}:
    raise ValueError(f'Unsupported TABFM_DEVICE={TABFM_DEVICE_PREF}')

SAMPLE_TRAIN_ROWS = int(os.getenv('KKBOX_SAMPLE_TRAIN_ROWS', '120000'))
SAMPLE_EVAL_ROWS = int(os.getenv('KKBOX_SAMPLE_EVAL_ROWS', '120000'))
TABFM_CONTEXT_MAX_ROWS = int(os.getenv('TABFM_CONTEXT_MAX_ROWS', '1500'))
TABFM_EVAL_MAX_ROWS = int(os.getenv('TABFM_EVAL_MAX_ROWS', '1000'))
TABFM_FAST_MODE = os.getenv('TABFM_FAST_MODE', '0').strip() == '1'

MIN_SAMPLE_ROWS = int(os.getenv('KKBOX_MIN_SAMPLE_ROWS', '2000'))
if SAMPLE_TRAIN_ROWS != 0 and SAMPLE_TRAIN_ROWS <= MIN_SAMPLE_ROWS:
    raise ValueError(
        f'Sample row settings are too small for a reliable benchmark. '
        f'Use 0 (full) or > {MIN_SAMPLE_ROWS} rows for train.'
    )
if SAMPLE_EVAL_ROWS != 0 and SAMPLE_EVAL_ROWS <= MIN_SAMPLE_ROWS:
    raise ValueError(
        f'Sample row settings are too small for a reliable benchmark. '
        f'Use 0 (full) or > {MIN_SAMPLE_ROWS} rows for eval.'
    )
if TABFM_CONTEXT_MAX_ROWS <= 500:
    raise ValueError('TABFM_CONTEXT_MAX_ROWS is too small to train stable TabFM variants.')
if TABFM_EVAL_MAX_ROWS != 0 and TABFM_EVAL_MAX_ROWS <= 500:
    raise ValueError('TABFM_EVAL_MAX_ROWS must be 0 (full) or > 500.')


def resolve_tabfm_device(preference: str) -> str:
    if preference == 'auto':
        return 'cuda' if torch.cuda.is_available() else 'cpu'
    if preference == 'cuda' and not torch.cuda.is_available():
        logger.warning('TABFM_DEVICE=cuda requested but CUDA unavailable; falling back to cpu')
        return 'cpu'
    return preference


def find_project_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / 'pyproject.toml').exists():
            return cand
    raise RuntimeError('Could not find project root (pyproject.toml not found).')


PROJECT_ROOT = find_project_root(Path.cwd())
PROBLEM_ROOT = PROJECT_ROOT / 'problems' / 'problem2_saas_subscription_churn'
DATA_DIR = PROBLEM_ROOT / 'data' / 'raw'
ARTIFACT_DIR = PROBLEM_ROOT / 'artifacts'
MODEL_CACHE_DIR = PROBLEM_ROOT / 'data' / 'models' / 'google-tabfm-1.0.0-pytorch'

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = resolve_tabfm_device(TABFM_DEVICE_PREF)

logger.info('Project root: {}', PROJECT_ROOT)
logger.info('Problem root: {}', PROBLEM_ROOT)
logger.info('Data dir: {}', DATA_DIR)
logger.info('Artifact dir: {}', ARTIFACT_DIR)
logger.info('Sample rows train/eval: {}/{}', SAMPLE_TRAIN_ROWS, SAMPLE_EVAL_ROWS)
logger.info('TabFM context max rows: {}', TABFM_CONTEXT_MAX_ROWS)
logger.info('Evaluation max rows per split (0 means full): {}', TABFM_EVAL_MAX_ROWS)
logger.info('TabFM fast mode: {}', TABFM_FAST_MODE)
logger.info('TabFM device preference={} effective={}', TABFM_DEVICE_PREF, DEVICE)

sns.set_theme(style='whitegrid')


## 2) Download KKBox competition data (idempotent)

This cell downloads only the required `.7z` files from Kaggle if missing, then extracts CSVs with `7z`.

Required files:
- `train_v2.csv.7z`
- `transactions_v2.csv.7z`
- `user_logs_v2.csv.7z`
- `members_v3.csv.7z`


In [ ]:
COMPETITION = 'kkbox-churn-prediction-challenge'
REQUIRED_ARCHIVES = [
    'train_v2.csv.7z',
    'transactions_v2.csv.7z',
    'user_logs_v2.csv.7z',
    'members_v3.csv.7z',
]
REQUIRED_CSVS = [a.removesuffix('.7z') for a in REQUIRED_ARCHIVES]


def run_command(cmd: list[str]) -> None:
    logger.info('Running: {}', ' '.join(cmd))
    subprocess.run(cmd, check=True)


def ensure_7z() -> None:
    if shutil.which('7z') is None:
        raise RuntimeError('7z is required but not available on PATH.')


def csv_exists(csv_name: str) -> bool:
    p = DATA_DIR / csv_name
    return p.exists() and p.stat().st_size > 0


ensure_7z()

missing_csvs = [c for c in REQUIRED_CSVS if not csv_exists(c)]
if missing_csvs:
    if shutil.which('kaggle') is None:
        raise RuntimeError('kaggle CLI is not installed. Install and authenticate first.')

    for archive in REQUIRED_ARCHIVES:
        csv_name = archive.removesuffix('.7z')
        if csv_exists(csv_name):
            logger.info('Using cached CSV: {}', csv_name)
            continue

        archive_path = DATA_DIR / archive
        if not archive_path.exists() or archive_path.stat().st_size == 0:
            run_command([
                'kaggle',
                'competitions',
                'download',
                '-c',
                COMPETITION,
                '-f',
                archive,
                '-p',
                str(DATA_DIR),
            ])

        run_command(['7z', 'x', str(archive_path), f'-o{DATA_DIR}', '-y'])

missing_after = [c for c in REQUIRED_CSVS if not csv_exists(c)]
if missing_after:
    raise FileNotFoundError(f'Missing CSVs after download/extract: {missing_after}')

sorted(p.name for p in DATA_DIR.glob('*.csv'))


## 3) Quick data audit and schema

We verify row counts and key date ranges to confirm the intended temporal window.


In [ ]:
DATA_FILES = {
    'train_v2': DATA_DIR / 'train_v2.csv',
    'transactions_v2': DATA_DIR / 'transactions_v2.csv',
    'user_logs_v2': DATA_DIR / 'user_logs_v2.csv',
    'members_v3': DATA_DIR / 'members_v3.csv',
}

for k, p in DATA_FILES.items():
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

row_counts = {
    name: pl.scan_csv(path, infer_schema=False).select(pl.len().alias('n')).collect(engine='streaming').item()
    for name, path in DATA_FILES.items()
}

date_ranges = {
    'user_logs_v2': pl.scan_csv(DATA_FILES['user_logs_v2'], schema_overrides={'date': pl.Int32})
        .select(pl.col('date').min().alias('min_date'), pl.col('date').max().alias('max_date'))
        .collect(engine='streaming')
        .to_dict(as_series=False),
    'transactions_v2': pl.scan_csv(
            DATA_FILES['transactions_v2'],
            schema_overrides={'transaction_date': pl.Int32, 'membership_expire_date': pl.Int32},
        )
        .select(
            pl.col('transaction_date').min().alias('min_tx_date'),
            pl.col('transaction_date').max().alias('max_tx_date'),
            pl.col('membership_expire_date').min().alias('min_exp_date'),
            pl.col('membership_expire_date').max().alias('max_exp_date'),
        )
        .collect(engine='streaming')
        .to_dict(as_series=False),
}

print('Row counts:')
for k, v in row_counts.items():
    print(f'  {k}: {v:,}')

print('\nDate ranges:')
for k, d in date_ranges.items():
    print(f'  {k}: {d}')


## 4) Label sampling (stratified, disjoint train/eval pools)

We use `train_v2` only and create disjoint pools:

- Train labels: stratified sample (`KKBOX_SAMPLE_TRAIN_ROWS`)
- Eval labels: stratified sample from the remaining users (`KKBOX_SAMPLE_EVAL_ROWS`)


In [ ]:
labels_full = (
    pl.read_csv(
        DATA_FILES['train_v2'],
        schema_overrides={'msno': pl.String, 'is_churn': pl.Int8},
    )
    .select('msno', 'is_churn')
    .unique('msno')
)


@dataclass
class LabelPool:
    train_labels: pl.DataFrame
    eval_labels: pl.DataFrame


def stratified_sample(labels: pl.DataFrame, max_rows: int, seed: int) -> pl.DataFrame:
    if max_rows == 0 or labels.height <= max_rows:
        return labels

    counts_df = labels.group_by('is_churn').len().sort('is_churn')
    counts = counts_df.to_dict(as_series=False)
    classes = counts['is_churn']
    class_counts = counts['len']
    total = sum(class_counts)

    sampled_parts: list[pl.DataFrame] = []
    allocated = 0

    for cls, cls_count in zip(classes, class_counts):
        n = max(1, int(round(max_rows * (cls_count / total))))
        part = (
            labels
            .filter(pl.col('is_churn') == cls)
            .sample(n=min(n, cls_count), seed=seed + int(cls), with_replacement=False)
        )
        sampled_parts.append(part)
        allocated += part.height

    sampled = pl.concat(sampled_parts, how='vertical').unique('msno')

    if sampled.height > max_rows:
        sampled = sampled.sample(n=max_rows, seed=seed, with_replacement=False)

    if sampled.height < max_rows:
        remainder = labels.join(sampled.select('msno'), on='msno', how='anti')
        need = min(max_rows - sampled.height, remainder.height)
        if need > 0:
            sampled = pl.concat(
                [sampled, remainder.sample(n=need, seed=seed + 999, with_replacement=False)],
                how='vertical',
            ).unique('msno')

    return sampled




def stratified_train_eval_split(labels: pl.DataFrame, train_fraction: float, seed: int) -> tuple[pl.DataFrame, pl.DataFrame]:
    counts_df = labels.group_by('is_churn').len().sort('is_churn')
    counts = counts_df.to_dict(as_series=False)

    train_parts: list[pl.DataFrame] = []
    eval_parts: list[pl.DataFrame] = []

    for cls, cls_count in zip(counts['is_churn'], counts['len']):
        class_rows = labels.filter(pl.col('is_churn') == cls)

        if cls_count <= 1:
            train_parts.append(class_rows)
            continue

        n_train = int(round(cls_count * train_fraction))
        n_train = max(1, min(cls_count - 1, n_train))

        train_part = class_rows.sample(n=n_train, seed=seed + int(cls), with_replacement=False)
        eval_part = class_rows.join(train_part.select('msno'), on='msno', how='anti')

        train_parts.append(train_part)
        eval_parts.append(eval_part)

    train_labels = pl.concat(train_parts, how='vertical').unique('msno')
    eval_labels = pl.concat(eval_parts, how='vertical').unique('msno') if eval_parts else pl.DataFrame(schema={'msno': pl.String, 'is_churn': pl.Int8})

    return train_labels, eval_labels

def build_label_pools(labels: pl.DataFrame, n_train: int, n_eval: int, seed: int) -> LabelPool:
    if n_train == 0 and n_eval == 0:
        train_labels, eval_labels = stratified_train_eval_split(labels, train_fraction=0.70, seed=seed)
        logger.info('Using full-label stratified split for KKBox: train={} eval={}', train_labels.height, eval_labels.height)
        return LabelPool(train_labels=train_labels, eval_labels=eval_labels)

    if n_train == 0 and n_eval > 0:
        eval_labels = stratified_sample(labels, min(n_eval, labels.height), seed=seed + 1)
        train_labels = labels.join(eval_labels.select('msno'), on='msno', how='anti')
        return LabelPool(train_labels=train_labels, eval_labels=eval_labels)

    train_labels = stratified_sample(labels, min(n_train, labels.height), seed=seed)
    remainder = labels.join(train_labels.select('msno'), on='msno', how='anti')

    if n_eval == 0:
        eval_labels = remainder
    else:
        eval_labels = stratified_sample(remainder, min(n_eval, remainder.height), seed=seed + 1)

    if n_eval > 0 and eval_labels.height < int(0.6 * n_eval):
        logger.warning(
            'Eval pool is smaller than requested due to disjoint sampling limits: requested={} got={}',
            n_eval,
            eval_labels.height,
        )

    return LabelPool(train_labels=train_labels, eval_labels=eval_labels)


label_pools = build_label_pools(labels_full, SAMPLE_TRAIN_ROWS, SAMPLE_EVAL_ROWS, seed=SEED)

print('Label pool sizes:')
print('  train_labels:', label_pools.train_labels.shape)
print('  eval_labels :', label_pools.eval_labels.shape)
print('  churn rate train/eval:',
      round(label_pools.train_labels['is_churn'].mean(), 4),
      round(label_pools.eval_labels['is_churn'].mean(), 4))


## 5) Feature engineering (streaming, leakage-safe)

Feature cutoff is fixed at **2017-03-31** for both train/eval pools (all from `train_v2` universe).

Feature families:
- Member profile and account tenure
- Transactions in the recent history window
- Product usage and engagement from user logs


In [ ]:
FEATURE_CUTOFF = date(2017, 3, 31)
TX_LOOKBACK_DAYS = 180
LOG_LOOKBACK_DAYS = 30

TX_SCHEMA = {
    'msno': pl.String,
    'payment_method_id': pl.Int32,
    'payment_plan_days': pl.Int32,
    'plan_list_price': pl.Float64,
    'actual_amount_paid': pl.Float64,
    'is_auto_renew': pl.Int8,
    'transaction_date': pl.Int32,
    'membership_expire_date': pl.Int32,
    'is_cancel': pl.Int8,
}

LOG_SCHEMA = {
    'msno': pl.String,
    'date': pl.Int32,
    'num_25': pl.Int64,
    'num_50': pl.Int64,
    'num_75': pl.Int64,
    'num_985': pl.Int64,
    'num_100': pl.Int64,
    'num_unq': pl.Int64,
    'total_secs': pl.Float64,
}

MEMBER_SCHEMA = {
    'msno': pl.String,
    'city': pl.Int32,
    'bd': pl.Int32,
    'gender': pl.String,
    'registered_via': pl.Int32,
    'registration_init_time': pl.Int32,
}


def int_yyyymmdd_to_date(expr: pl.Expr) -> pl.Expr:
    return expr.cast(pl.Int64).cast(pl.Utf8).str.strptime(pl.Date, format='%Y%m%d', strict=False)


def safe_ratio_expr(numerator: pl.Expr, denominator: pl.Expr, alias: str) -> pl.Expr:
    return pl.when(denominator > 0).then(numerator / denominator).otherwise(None).alias(alias)


def build_member_features(label_msno: pl.DataFrame, cutoff_date: date) -> pl.DataFrame:
    cutoff = pl.lit(cutoff_date)
    return (
        pl.scan_csv(DATA_FILES['members_v3'], schema_overrides=MEMBER_SCHEMA, infer_schema_length=1000, low_memory=True)
        .join(label_msno.lazy(), on='msno', how='inner')
        .select('msno', 'city', 'bd', 'gender', 'registered_via', 'registration_init_time')
        .with_columns(
            int_yyyymmdd_to_date(pl.col('registration_init_time')).alias('registration_init_date'),
            pl.when((pl.col('bd') >= 10) & (pl.col('bd') <= 90)).then(pl.col('bd')).otherwise(None).alias('age_clean'),
        )
        .with_columns(
            (cutoff - pl.col('registration_init_date')).dt.total_days().alias('account_tenure_days'),
            pl.col('city').cast(pl.Utf8).alias('city'),
            pl.col('gender').cast(pl.Utf8).fill_null('unknown').alias('gender'),
            pl.col('registered_via').cast(pl.Utf8).alias('registered_via'),
        )
        .select('msno', 'city', 'gender', 'registered_via', 'age_clean', 'account_tenure_days')
        .collect(engine='streaming')
    )


def build_transaction_features(label_msno: pl.DataFrame, cutoff_date: date) -> pl.DataFrame:
    cutoff = pl.lit(cutoff_date)
    min_date = cutoff_date - timedelta(days=TX_LOOKBACK_DAYS)

    tx_lf = (
        pl.scan_csv(DATA_FILES['transactions_v2'], schema_overrides=TX_SCHEMA, infer_schema_length=1000, low_memory=True)
        .join(label_msno.lazy(), on='msno', how='inner')
        .select(
            'msno',
            'payment_method_id',
            'payment_plan_days',
            'plan_list_price',
            'actual_amount_paid',
            'is_auto_renew',
            'transaction_date',
            'membership_expire_date',
            'is_cancel',
        )
        .with_columns(
            int_yyyymmdd_to_date(pl.col('transaction_date')).alias('transaction_date'),
            int_yyyymmdd_to_date(pl.col('membership_expire_date')).alias('membership_expire_date'),
        )
        .filter(pl.col('transaction_date').is_not_null())
        .filter(pl.col('transaction_date') <= cutoff)
        .filter(pl.col('transaction_date') >= pl.lit(min_date))
        .with_columns((cutoff - pl.col('transaction_date')).dt.total_days().alias('days_since_tx'))
    )

    return (
        tx_lf.group_by('msno')
        .agg(
            pl.len().alias('txn_count_180d'),
            pl.col('actual_amount_paid').mean().alias('txn_amount_mean_180d'),
            pl.col('actual_amount_paid').sum().alias('txn_amount_sum_180d'),
            pl.col('payment_plan_days').mean().alias('plan_days_mean_180d'),
            pl.col('is_auto_renew').mean().alias('auto_renew_rate_180d'),
            pl.col('is_cancel').mean().alias('cancel_rate_180d'),
            pl.col('payment_method_id').n_unique().alias('payment_method_nunique_180d'),
            pl.col('transaction_date').max().alias('last_transaction_date'),
            pl.col('membership_expire_date').sort_by('transaction_date').last().alias('latest_membership_expire_date'),
            pl.col('payment_plan_days').sort_by('transaction_date').last().alias('last_payment_plan_days'),
            pl.col('actual_amount_paid').sort_by('transaction_date').last().alias('last_actual_amount_paid'),
            pl.col('plan_list_price').sort_by('transaction_date').last().alias('last_plan_list_price'),
            pl.col('is_auto_renew').sort_by('transaction_date').last().alias('last_is_auto_renew'),
            pl.col('is_cancel').sort_by('transaction_date').last().alias('last_is_cancel'),
        )
        .with_columns(
            (cutoff - pl.col('last_transaction_date')).dt.total_days().alias('days_since_last_transaction'),
            (pl.col('latest_membership_expire_date') - cutoff).dt.total_days().alias('days_to_latest_expire'),
        )
        .collect(engine='streaming')
    )


def build_log_features(label_msno: pl.DataFrame, cutoff_date: date) -> pl.DataFrame:
    cutoff = pl.lit(cutoff_date)
    min_date = cutoff_date - timedelta(days=LOG_LOOKBACK_DAYS)

    logs_lf = (
        pl.scan_csv(DATA_FILES['user_logs_v2'], schema_overrides=LOG_SCHEMA, infer_schema_length=1000, low_memory=True)
        .join(label_msno.lazy(), on='msno', how='inner')
        .select('msno', 'date', 'num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq', 'total_secs')
        .with_columns(int_yyyymmdd_to_date(pl.col('date')).alias('date'))
        .filter(pl.col('date').is_not_null())
        .filter(pl.col('date') <= cutoff)
        .filter(pl.col('date') >= pl.lit(min_date))
        .with_columns(
            (cutoff - pl.col('date')).dt.total_days().alias('days_since_log'),
            (pl.col('num_100') + pl.col('num_985')).alias('full_play_count'),
            (pl.col('num_25') + pl.col('num_50')).alias('skip_play_count'),
            (
                pl.col('num_25')
                + pl.col('num_50')
                + pl.col('num_75')
                + pl.col('num_985')
                + pl.col('num_100')
            ).alias('total_play_count'),
        )
    )

    return (
        logs_lf.group_by('msno')
        .agg(
            pl.col('date').n_unique().alias('login_days_30d'),
            pl.col('total_secs').sum().alias('total_secs_30d'),
            pl.col('num_unq').sum().alias('unique_songs_30d'),
            pl.col('full_play_count').sum().alias('full_plays_30d'),
            pl.col('skip_play_count').sum().alias('skip_plays_30d'),
            pl.col('total_play_count').sum().alias('plays_30d'),
            pl.col('date').max().alias('last_log_date'),
        )
        .with_columns(
            (cutoff - pl.col('last_log_date')).dt.total_days().alias('days_since_last_log'),
            safe_ratio_expr(pl.col('full_plays_30d'), pl.col('plays_30d'), 'completion_ratio_30d'),
            safe_ratio_expr(pl.col('skip_plays_30d'), pl.col('plays_30d'), 'skip_ratio_30d'),
        )
        .collect(engine='streaming')
    )


def build_feature_table(labels_df: pl.DataFrame, cutoff_date: date) -> pl.DataFrame:
    label_msno = labels_df.select('msno').unique('msno')
    member_features = build_member_features(label_msno, cutoff_date)
    tx_features = build_transaction_features(label_msno, cutoff_date)
    log_features = build_log_features(label_msno, cutoff_date)

    return (
        labels_df
        .join(member_features, on='msno', how='left')
        .join(tx_features, on='msno', how='left')
        .join(log_features, on='msno', how='left')
        .with_columns(pl.lit(cutoff_date).cast(pl.Date).alias('feature_cutoff_date'))
    )


## 6) Build train/eval feature tables


In [ ]:
t0 = time.time()
train_features_pl = build_feature_table(label_pools.train_labels, FEATURE_CUTOFF)
logger.info('Built train features: {} in {:.1f}s', train_features_pl.shape, time.time() - t0)

t1 = time.time()
eval_features_pl = build_feature_table(label_pools.eval_labels, FEATURE_CUTOFF)
logger.info('Built eval features: {} in {:.1f}s', eval_features_pl.shape, time.time() - t1)

feature_null_snapshot = (
    train_features_pl
    .select([
        pl.col(c).null_count().alias(c)
        for c in ['txn_count_180d', 'login_days_30d', 'total_secs_30d', 'days_since_last_transaction']
        if c in train_features_pl.columns
    ])
)
feature_null_snapshot


## 7) Convert to modeling frames and split validation/test

- Train pool: from sampled train labels
- Eval pool: split into validation and test (stratified 50/50)


In [ ]:
ID_COLS = ['msno', 'feature_cutoff_date']


def polars_to_pandas_safe(df: pl.DataFrame) -> pd.DataFrame:
    try:
        return df.to_pandas()
    except ModuleNotFoundError:
        # Fallback path if pyarrow is unavailable.
        return pd.DataFrame(df.to_dict(as_series=False))


def cast_for_memory(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        s = out[col]
        if pd.api.types.is_float_dtype(s):
            out[col] = pd.to_numeric(s, downcast='float')
        elif pd.api.types.is_integer_dtype(s):
            out[col] = pd.to_numeric(s, downcast='integer')
    return out


def make_model_xy(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, pd.DataFrame]:
    y = df['is_churn'].astype(int).to_numpy()
    meta = df[[c for c in ID_COLS if c in df.columns]].copy()
    X = df.drop(columns=['is_churn'] + [c for c in ID_COLS if c in df.columns], errors='ignore').copy()
    for c in X.columns:
        if pd.api.types.is_datetime64_any_dtype(X[c]):
            X[c] = X[c].astype('string')
        if isinstance(X[c].dtype, pd.api.extensions.ExtensionDtype):
            X[c] = X[c].astype(object)
    X = X.where(pd.notna(X), np.nan)
    return X, y, meta


train_df = cast_for_memory(polars_to_pandas_safe(train_features_pl))
eval_df = cast_for_memory(polars_to_pandas_safe(eval_features_pl))

for c in ['feature_cutoff_date', 'last_transaction_date', 'latest_membership_expire_date', 'last_log_date']:
    if c in train_df.columns:
        train_df[c] = pd.to_datetime(train_df[c], errors='coerce')
    if c in eval_df.columns:
        eval_df[c] = pd.to_datetime(eval_df[c], errors='coerce')

if eval_df['is_churn'].nunique() < 2:
    raise RuntimeError('Eval pool has single class, cannot split for binary evaluation.')

X_eval = eval_df.drop(columns=['is_churn'])
y_eval = eval_df['is_churn'].astype(int).to_numpy()
X_val, X_test, y_val, y_test = train_test_split(
    X_eval,
    y_eval,
    test_size=0.5,
    random_state=SEED,
    stratify=y_eval,
)
val_df = X_val.copy()
val_df['is_churn'] = y_val
test_df = X_test.copy()
test_df['is_churn'] = y_test

print('Rows train/val/test:', len(train_df), len(val_df), len(test_df))
print(
    'Churn rates train/val/test:',
    round(train_df['is_churn'].mean(), 4),
    round(val_df['is_churn'].mean(), 4),
    round(test_df['is_churn'].mean(), 4),
)


## 8) Modeling helpers and XGBoost baseline


In [ ]:
def evaluate_classifier(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.5) -> dict[str, float]:
    y_pred = (y_score >= threshold).astype(int)
    return {
        'roc_auc': float(roc_auc_score(y_true, y_score)),
        'pr_auc': float(average_precision_score(y_true, y_score)),
        'brier': float(brier_score_loss(y_true, y_score)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
    }


def train_xgboost_baseline(X_train: pd.DataFrame, y_train: np.ndarray) -> Pipeline:
    cat_cols = X_train.select_dtypes(include=['object', 'category', 'string', 'bool']).columns.tolist()
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                'num',
                Pipeline([('imputer', SimpleImputer(strategy='median'))]),
                num_cols,
            ),
            (
                'cat',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore')),
                ]),
                cat_cols,
            ),
        ]
    )

    xgb = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=2,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',
    )

    model = Pipeline([('preprocessor', preprocessor), ('model', xgb)])
    model.fit(X_train, y_train)
    return model


## 9) TabFM checkpoint management (first-run download)

The first run may need to download TabFM classification weights from Hugging Face.

Official references:
- Google Research blog: https://research.google/blog/introducing-tabfm-a-zero-shot-foundation-model-for-tabular-data/
- TabFM repository: https://github.com/google-research/tabfm
- TabFM PyTorch weights: https://huggingface.co/google/tabfm-1.0.0-pytorch


In [ ]:
TABFM_CHECKPOINT_OVERRIDE = os.getenv('TABFM_CHECKPOINT_PATH', '').strip()


def ensure_tabfm_checkpoint() -> Path:
    if TABFM_CHECKPOINT_OVERRIDE:
        override_path = Path(TABFM_CHECKPOINT_OVERRIDE)
        if override_path.is_file() and override_path.name == 'pytorch_model.bin':
            return override_path
        if override_path.is_dir():
            cand = override_path / 'classification' / 'pytorch_model.bin'
            if cand.exists():
                return cand
            cand2 = override_path / 'pytorch_model.bin'
            if cand2.exists():
                return cand2
        raise FileNotFoundError(f'Invalid TABFM_CHECKPOINT_PATH: {TABFM_CHECKPOINT_OVERRIDE}')

    local_ckpt = MODEL_CACHE_DIR / 'classification' / 'pytorch_model.bin'
    if local_ckpt.exists() and local_ckpt.stat().st_size > 0:
        logger.info('Using cached TabFM checkpoint: {}', local_ckpt)
        return local_ckpt

    logger.warning('TabFM checkpoint not found locally. Downloading classification weights (first run may be large).')
    os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
    os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '300')
    os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '300')

    from huggingface_hub import hf_hub_download

    downloaded_path = hf_hub_download(
        repo_id='google/tabfm-1.0.0-pytorch',
        filename='classification/pytorch_model.bin',
        local_dir=str(MODEL_CACHE_DIR),
    )
    ckpt_path = Path(downloaded_path)
    logger.info('Downloaded TabFM checkpoint to {} ({:.2f} MB)', ckpt_path, ckpt_path.stat().st_size / (1024**2))
    return ckpt_path


TABFM_CKPT_PATH = ensure_tabfm_checkpoint()
TABFM_CKPT_PATH


## 10) Train XGBoost + 3 TabFM variants

To keep runtime stable, TabFM is fit on a stratified context subset (`TABFM_CONTEXT_MAX_ROWS`) and model ranking metrics are computed on a stratified evaluation slice (`TABFM_EVAL_MAX_ROWS`). The selected champion is then rescored on the full validation/test splits for business decisions.


In [ ]:
X_train, y_train, train_meta = make_model_xy(train_df)
X_val, y_val, val_meta = make_model_xy(val_df)
X_test, y_test, test_meta = make_model_xy(test_df)


def load_tabfm_backbone(device: str) -> Any:
    ckpt_root = TABFM_CKPT_PATH.parent.parent if TABFM_CKPT_PATH.parent.name == 'classification' else TABFM_CKPT_PATH.parent
    return tabfm_v1_0_0.load(
        model_type='classification',
        checkpoint_path=str(ckpt_root),
        device=device,
    )


def pick_tabfm_device(requested: str) -> str:
    if requested.startswith('cuda') and torch.cuda.is_available():
        total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        logger.info('Detected GPU memory {:.2f} GiB', total_mem_gb)
        if total_mem_gb < 12:
            logger.warning('GPU memory <12 GiB; forcing CPU for TabFM stability')
            return 'cpu'
    return requested


def fit_tabfm_variants(X: pd.DataFrame, y: np.ndarray, requested_device: str) -> dict[str, TabFMClassifier]:
    device = pick_tabfm_device(requested_device)

    if len(X) > TABFM_CONTEXT_MAX_ROWS:
        X_fit, _, y_fit, _ = train_test_split(
            X,
            y,
            train_size=TABFM_CONTEXT_MAX_ROWS,
            random_state=SEED,
            stratify=y,
        )
        logger.warning('TabFM context capped: {} -> {} rows', len(X), len(X_fit))
    else:
        X_fit, y_fit = X, y

    batch_size = 1 if device == 'cpu' else 2
    models: dict[str, TabFMClassifier] = {}

    default_estimators = 1 if TABFM_FAST_MODE else 4
    ensemble_estimators = 1 if TABFM_FAST_MODE else 6
    advanced_estimators = 1 if TABFM_FAST_MODE else 6
    advanced_norm_methods = ['none'] if TABFM_FAST_MODE else ['none', 'power', 'quantile_rtdl']
    advanced_n_feature_crosses = 0 if TABFM_FAST_MODE else 'sqrt'
    advanced_n_svd_features = 0 if TABFM_FAST_MODE else 'sqrt'
    advanced_total_svd_pool = 32 if TABFM_FAST_MODE else 128

    if TABFM_FAST_MODE:
        logger.warning('TABFM_FAST_MODE=1 enabled; using reduced TabFM estimator complexity for smoke execution')

    m_default = TabFMClassifier(
        model=load_tabfm_backbone(device),
        n_estimators=default_estimators,
        batch_size=batch_size,
        random_state=SEED,
        n_feature_crosses=0,
        n_svd_features=0,
        enable_nnls=False,
        binary_calibration_method=None,
        multiclass_calibration_method=None,
        verbose=False,
    )
    m_default.fit(X_fit, y_fit)
    models['tabfm_default'] = m_default

    m_ensemble = TabFMClassifier.ensemble(
        model=load_tabfm_backbone(device),
        n_estimators=ensemble_estimators,
        batch_size=batch_size,
        random_state=SEED,
        num_folds_for_cv=2,
        min_rows_for_single_val_split=100,
        verbose=False,
    )
    m_ensemble.fit(X_fit, y_fit)
    models['tabfm_ensemble_preset'] = m_ensemble

    m_advanced = TabFMClassifier(
        model=load_tabfm_backbone(device),
        n_estimators=advanced_estimators,
        norm_methods=advanced_norm_methods,
        feat_shuffle_method='random',
        class_shift=True,
        permute_categorical=True,
        n_feature_crosses=advanced_n_feature_crosses,
        n_svd_features=advanced_n_svd_features,
        total_svd_pool=advanced_total_svd_pool,
        average_logits=False,
        enable_nnls=False,
        nnls_beta=0.75,
        binary_calibration_method=None,
        random_state=SEED,
        batch_size=batch_size,
        num_folds_for_cv=2,
        min_rows_for_single_val_split=100,
        verbose=False,
    )
    m_advanced.fit(X_fit, y_fit)
    models['tabfm_advanced_custom'] = m_advanced

    logger.info('TabFM variants trained on device={}', device)
    return models


def get_scores(model: Any, X: pd.DataFrame) -> np.ndarray:
    proba = np.asarray(model.predict_proba(X))
    if proba.ndim != 2 or proba.shape[1] != 2:
        raise ValueError(f'Expected binary predict_proba output shape [N,2], got {proba.shape}')
    return proba[:, 1].astype(float)


t0 = time.time()
xgb_model = train_xgboost_baseline(X_train, y_train)
logger.info('XGBoost trained in {:.1f}s', time.time() - t0)

t1 = time.time()
tabfm_models = fit_tabfm_variants(X_train, y_train, requested_device=DEVICE)
logger.info('TabFM variants trained in {:.1f}s', time.time() - t1)


## 11) Evaluate models


In [ ]:
def build_eval_slice(X: pd.DataFrame, y: np.ndarray, max_rows: int, seed_offset: int) -> tuple[pd.DataFrame, np.ndarray]:
    if max_rows == 0 or len(X) <= max_rows:
        return X, y

    stratify_target = y if len(np.unique(y)) > 1 else None
    _, X_sampled, _, y_sampled = train_test_split(
        X,
        y,
        test_size=max_rows,
        random_state=SEED + seed_offset,
        stratify=stratify_target,
    )
    return X_sampled, y_sampled


X_val_eval, y_val_eval = build_eval_slice(X_val, y_val, TABFM_EVAL_MAX_ROWS, seed_offset=101)
X_test_eval, y_test_eval = build_eval_slice(X_test, y_test, TABFM_EVAL_MAX_ROWS, seed_offset=202)
logger.info('Evaluation rows used (val/test): {}/{}', len(X_val_eval), len(X_test_eval))

model_registry: dict[str, Any] = {'xgboost_baseline': xgb_model, **tabfm_models}
rows: list[dict[str, Any]] = []
predictions: dict[str, dict[str, np.ndarray]] = {}

for model_name, model in model_registry.items():
    val_scores = get_scores(model, X_val_eval)
    test_scores = get_scores(model, X_test_eval)
    predictions[model_name] = {'val': val_scores, 'test': test_scores}

    rows.append({'model': model_name, 'split': 'val', **evaluate_classifier(y_val_eval, val_scores)})
    rows.append({'model': model_name, 'split': 'test', **evaluate_classifier(y_test_eval, test_scores)})

metrics_df = pd.DataFrame(rows).sort_values(['split', 'pr_auc'], ascending=[True, False]).reset_index(drop=True)
val_rank = metrics_df[metrics_df['split'] == 'val'].sort_values('pr_auc', ascending=False).reset_index(drop=True)
champion_model_name = val_rank.loc[0, 'model']
logger.info('Champion model on validation PR-AUC: {}', champion_model_name)

# Re-score champion on full splits for downstream business decisions.
champion_model = model_registry[champion_model_name]
champion_val_scores = get_scores(champion_model, X_val)
champion_test_scores = get_scores(champion_model, X_test)
logger.info('Champion full scoring rows (val/test): {}/{}', len(champion_val_scores), len(champion_test_scores))

metrics_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_df = metrics_df[metrics_df['split'] == 'test'].sort_values('pr_auc', ascending=False)

sns.barplot(data=plot_df, x='pr_auc', y='model', ax=axes[0], palette='viridis')
axes[0].set_title('Test PR-AUC')
axes[0].set_xlabel('PR-AUC')
axes[0].set_ylabel('')

sns.barplot(data=plot_df, x='roc_auc', y='model', ax=axes[1], palette='mako')
axes[1].set_title('Test ROC-AUC')
axes[1].set_xlabel('ROC-AUC')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


In [ ]:
if len(champion_test_scores) != len(y_test):
    raise ValueError(
        f'Champion test score length mismatch: len(scores)={len(champion_test_scores)} len(y_test)={len(y_test)}'
    )

prob_true, prob_pred = calibration_curve(y_test, champion_test_scores, n_bins=10, strategy='quantile')

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
plt.plot(prob_pred, prob_true, marker='o', label=champion_model_name)
plt.title('Calibration Curve (Test)')
plt.xlabel('Predicted churn probability')
plt.ylabel('Observed churn frequency')
plt.legend()
plt.tight_layout()
plt.show()


## 12) Retention action policy and campaign economics


In [ ]:
def risk_tier_from_score(score: float) -> str:
    if score >= 0.75:
        return 'high'
    if score >= 0.50:
        return 'medium'
    return 'low'


test_scored = test_meta.copy()
test_scored['is_churn'] = y_test
test_scored['churn_score'] = champion_test_scores

for col in ['login_days_30d', 'completion_ratio_30d', 'skip_ratio_30d', 'cancel_rate_180d', 'days_since_last_transaction']:
    if col in test_df.columns:
        test_scored[col] = test_df[col].values
    else:
        test_scored[col] = np.nan

test_scored['risk_tier'] = test_scored['churn_score'].map(risk_tier_from_score)

low_login_threshold = np.nanpercentile(test_scored['login_days_30d'], 35)
high_skip_threshold = np.nanpercentile(test_scored['skip_ratio_30d'], 70)
high_cancel_threshold = np.nanpercentile(test_scored['cancel_rate_180d'], 70)

test_scored['recommend_onboarding_session'] = np.where(
    (test_scored['risk_tier'].isin(['high', 'medium']))
    & ((test_scored['login_days_30d'] <= low_login_threshold) | (test_scored['skip_ratio_30d'] >= high_skip_threshold)),
    1,
    0,
)

test_scored['trigger_customer_success_intervention'] = np.where(
    (test_scored['risk_tier'] == 'high') | (test_scored['cancel_rate_180d'] >= high_cancel_threshold),
    1,
    0,
)

test_scored['adoption_playbook'] = np.select(
    [
        test_scored['trigger_customer_success_intervention'] == 1,
        test_scored['recommend_onboarding_session'] == 1,
        test_scored['risk_tier'] == 'medium',
    ],
    [
        'CSM outreach + retention offer',
        'Guided onboarding + activation workshop',
        'Lifecycle nudges + product tips',
    ],
    default='Standard nurture',
)

(
    test_scored
    .groupby(['risk_tier', 'adoption_playbook'], dropna=False)
    .size()
    .reset_index(name='customers')
    .sort_values(['risk_tier', 'customers'], ascending=[True, False])
)


In [ ]:
def simulate_top_k_campaign(
    y_true: np.ndarray,
    y_score: np.ndarray,
    k_values: tuple[float, ...],
    contact_cost: float,
    save_success_prob: float,
    annual_value_per_saved_customer: float,
) -> pd.DataFrame:
    n = len(y_true)
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    total_churners = int(y_true.sum())

    rows: list[dict[str, float]] = []
    for k in k_values:
        n_target = max(1, int(round(n * k)))
        targeted = y_sorted[:n_target]
        tp = int(targeted.sum())
        expected_saved = tp * save_success_prob
        expected_revenue_saved = expected_saved * annual_value_per_saved_customer
        expected_cost = n_target * contact_cost
        expected_net = expected_revenue_saved - expected_cost

        rows.append(
            {
                'k_fraction': k,
                'target_customers': n_target,
                'true_churners_targeted': tp,
                'precision_at_k': tp / n_target,
                'recall_at_k': tp / max(1, total_churners),
                'expected_saved_customers': expected_saved,
                'expected_revenue_saved': expected_revenue_saved,
                'expected_campaign_cost': expected_cost,
                'expected_net_value': expected_net,
            }
        )

    return pd.DataFrame(rows)


def optimize_cost_sensitive_threshold(
    y_true: np.ndarray,
    y_score: np.ndarray,
    contact_cost: float,
    save_success_prob: float,
    annual_value_per_saved_customer: float,
    grid_size: int = 301,
) -> tuple[pd.DataFrame, dict[str, float]]:
    thresholds = np.linspace(0.0, 1.0, grid_size)
    rows: list[dict[str, float]] = []

    for threshold in thresholds:
        targeted = (y_score >= threshold).astype(int)
        n_target = int(targeted.sum())
        tp = int(((targeted == 1) & (y_true == 1)).sum())

        expected_saved = tp * save_success_prob
        expected_revenue_saved = expected_saved * annual_value_per_saved_customer
        expected_cost = n_target * contact_cost
        expected_net = expected_revenue_saved - expected_cost

        rows.append(
            {
                'threshold': float(threshold),
                'targeted_customers': float(n_target),
                'true_churners_targeted': float(tp),
                'expected_saved_customers': float(expected_saved),
                'expected_revenue_saved': float(expected_revenue_saved),
                'expected_campaign_cost': float(expected_cost),
                'expected_net_value': float(expected_net),
            }
        )

    curve_df = pd.DataFrame(rows)
    best_idx = curve_df['expected_net_value'].idxmax()
    return curve_df, curve_df.loc[best_idx].to_dict()


BUSINESS_ASSUMPTIONS = {
    'contact_cost': 35.0,
    'save_success_prob': 0.30,
    'annual_value_per_saved_customer': 420.0,
}

topk_df = simulate_top_k_campaign(
    y_true=y_test,
    y_score=champion_test_scores,
    k_values=(0.05, 0.10, 0.15, 0.20, 0.30),
    **BUSINESS_ASSUMPTIONS,
)

threshold_curve_df, best_threshold_row = optimize_cost_sensitive_threshold(
    y_true=y_val,
    y_score=champion_val_scores,
    **BUSINESS_ASSUMPTIONS,
)

best_threshold = float(best_threshold_row['threshold'])
logger.info('Best economic threshold from validation: {:.3f}', best_threshold)

topk_df


In [ ]:
plt.figure(figsize=(8, 4))
sns.lineplot(data=threshold_curve_df, x='threshold', y='expected_net_value', linewidth=2)
plt.axvline(best_threshold, color='red', linestyle='--', label=f'best={best_threshold:.3f}')
plt.title('Expected net value vs threshold (validation)')
plt.xlabel('Threshold')
plt.ylabel('Expected net value')
plt.legend()
plt.tight_layout()
plt.show()


## 13) Persist artifacts


In [ ]:
metrics_path = ARTIFACT_DIR / 'problem2_kkbox_model_metrics.csv'
pred_path = ARTIFACT_DIR / 'problem2_kkbox_predictions_test.parquet'
intervention_path = ARTIFACT_DIR / 'problem2_kkbox_interventions.csv'
topk_path = ARTIFACT_DIR / 'problem2_kkbox_topk_campaign.csv'
threshold_curve_path = ARTIFACT_DIR / 'problem2_kkbox_threshold_curve.csv'
threshold_summary_path = ARTIFACT_DIR / 'problem2_kkbox_threshold_summary.csv'
runtime_meta_path = ARTIFACT_DIR / 'problem2_kkbox_runtime_meta.json'

pred_frame = test_scored[[
    'msno',
    'is_churn',
    'churn_score',
    'risk_tier',
    'recommend_onboarding_session',
    'trigger_customer_success_intervention',
    'adoption_playbook',
]].copy()

metrics_df.to_csv(metrics_path, index=False)

pred_frame_for_parquet = pred_frame.copy()
for col in ['is_churn', 'recommend_onboarding_session', 'trigger_customer_success_intervention']:
    if col in pred_frame_for_parquet.columns:
        pred_frame_for_parquet[col] = pred_frame_for_parquet[col].fillna(0).astype(int)

pred_pl = pl.DataFrame({col: pred_frame_for_parquet[col].tolist() for col in pred_frame_for_parquet.columns})
pred_pl.write_parquet(pred_path)

test_scored.to_csv(intervention_path, index=False)
topk_df.to_csv(topk_path, index=False)
threshold_curve_df.to_csv(threshold_curve_path, index=False)

threshold_summary_df = pd.DataFrame([
    {'split': 'validation', **best_threshold_row},
    {
        'split': 'test',
        **optimize_cost_sensitive_threshold(
            y_true=y_test,
            y_score=champion_test_scores,
            **BUSINESS_ASSUMPTIONS,
        )[1],
    },
])
threshold_summary_df.to_csv(threshold_summary_path, index=False)

runtime_meta = {
    'seed': SEED,
    'feature_cutoff': str(FEATURE_CUTOFF),
    'sample_train_rows': int(SAMPLE_TRAIN_ROWS),
    'sample_eval_rows': int(SAMPLE_EVAL_ROWS),
    'tabfm_context_max_rows': int(TABFM_CONTEXT_MAX_ROWS),
    'tabfm_eval_max_rows': int(TABFM_EVAL_MAX_ROWS),
    'tabfm_fast_mode': bool(TABFM_FAST_MODE),
    'tabfm_device_preference': TABFM_DEVICE_PREF,
    'tabfm_device_effective': DEVICE,
    'tabfm_checkpoint': str(TABFM_CKPT_PATH),
    'champion_model': champion_model_name,
    'business_assumptions': BUSINESS_ASSUMPTIONS,
}
runtime_meta_path.write_text(json.dumps(runtime_meta, indent=2))

for path in [
    metrics_path,
    pred_path,
    intervention_path,
    topk_path,
    threshold_curve_path,
    threshold_summary_path,
    runtime_meta_path,
]:
    logger.info('Wrote {}', path)

sorted(p.name for p in ARTIFACT_DIR.glob('problem2_kkbox_*'))


## 14) Final notes

### What was implemented
- End-to-end churn pipeline with stable memory profile.
- Baseline + 3 TabFM variants.
- Business-action outputs (risk tiers, interventions, ROI simulations).

### Production hardening recommendations
- Track metrics and artifacts in MLflow/W&B for each run.
- Add feature/data drift monitors for key usage and transaction signals.
- Trigger periodic re-training as subscription behavior shifts.
